In [1]:
from dataclasses import dataclass

import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import scipy.optimize as opt


In [2]:
@dataclass(frozen=True)
class Parameters:

    '''
    A dataclass to hold the parameters of a two-asset portfolio, containing
    one riskless asset and one risky asset. The risky asset has two possible 
    gross returns, gamma_heads and gamma_tails, which occur with probabilities
    p and 1 - p, respectively. The riskless asset has a gross return of r.
    The weight of the risky asset in the portfolio is b, and the weight of the
    riskless asset is 1 - b. The parameter alpha is used to determine the gross
    return of the risky asset in the tails state, which is given by 1/(alpha + gamma_heads).

    We consider the random variable 
    X = (gross_return_of_riskless_asset, gross_return_of_risky_asset), which is
    random because the gross return of the risky asset is random. The portfolio
    gross return S is given by the dot product of the weights vector [1 - b, b]
    and the returns vector [r, gamma_heads] if heads occurs, and by the dot 
    product of the weights vector [1 - b, b] and the returns vector
    [r, gamma_tails] if tails occurs. In other words, S = weights_vector . X,
    where the dot product is taken with the appropriate returns vector depending
    on whether heads or tails occurs.
    '''

    gamma_heads: float # gross returns of heads
    p: float # probability of heads
    alpha: float=0.0
    r: float=1.0 # gross returns of riskless asset

    @property
    def gamma_tails(self) -> float:
        '''Returns 1/(alpha * gamma_heads).'''
        return 1/(self.alpha*self.gamma_heads)
    @property
    def probabilities_vector(self) -> np.ndarray:
        '''Returns [1 - p, p].'''
        return np.array([1 - self.p, self.p])
    @property
    def gross_returns_vector_if_heads(self) -> np.ndarray:
        '''Returns X when X = [r, gamma_heads].'''
        return np.array([self.r, self.gamma_heads])
    @property
    def gross_returns_vector_if_tails(self) -> np.ndarray:
        '''Returns X when X = [r, gamma_tails].'''
        return np.array([self.r, self.gamma_tails])
    @property
    def mean_gross_returns_vector(self) -> np.ndarray:
        '''Returns mu, which is the mean of the gross returns vector, given by p*[r, gamma_heads] + (1 - p)*[r, gamma_tails].'''
        return self.p*self.gross_returns_vector_if_heads + (1 - self.p)*self.gross_returns_vector_if_tails

    def return_portfolio_gross_return_if_heads(self, weights_vector: np.ndarray) -> float:
        '''Returns the dot product of the weights_vector and the returns vector [r, gamma_heads].'''
        return np.dot(weights_vector, self.gross_returns_vector_if_heads)
    
    def return_portfolio_gross_return_if_tails(self, weights_vector: np.ndarray) -> float:
        '''Returns the dot product of the weights_vector and the returns vector [r, gamma_tails].'''
        return np.dot(weights_vector, self.gross_returns_vector_if_tails)        
    
    def return_log_portfolio_gross_returns(self, weights_vector: np.ndarray) -> np.ndarray:
        '''Returns the log of the portfolio gross returns for tails and heads, which is a length-2 vector.'''
        t = self.return_portfolio_gross_return_if_tails(weights_vector)
        h = self.return_portfolio_gross_return_if_heads(weights_vector)
        return np.log(np.array([t, h]))

    def return_expected_log_portfolio_gross_return(self, weights_vector: np.ndarray) -> float:
        '''Returns E[log(S)] = E[log(weights_vector . X)]'''
        return np.dot(self.probabilities_vector, self.return_log_portfolio_gross_returns(weights_vector))

    def return_growth_rate(self, weights_vector: np.ndarray) -> float:
        '''An alias for return_expected_log_portfolio_gross_return.'''
        return self.return_expected_log_portfolio_gross_return(weights_vector)

    def return_expected_portfolio_gross_return(self, weights_vector: np.ndarray) -> float:
        '''Returns E[S] = E[weights_vector . X] = weights_vector . E[X].'''
        return np.dot(weights_vector, self.mean_gross_returns_vector)       

    def return_arithmetic_portfolio_gross_return(self, weights_vector: np.ndarray) -> float:
        '''An alias for return_expected_portfolio_gross_return.'''
        return self.return_expected_portfolio_gross_return(weights_vector)

    def return_geometric_portfolio_gross_return(self, weights_vector: np.ndarray) -> float:
        '''Returns exp(E[log(S)])'''
        return np.exp(self.return_growth_rate(weights_vector))



In [3]:
def return_running_emperical_growth_rates(gross_returns_by_period: np.ndarray) -> np.ndarray:
    '''Returns the runningempirical growth rate, given by the average of the log of the gross returns by period.'''
    size = len(gross_returns_by_period)
    return np.cumsum(np.log(gross_returns_by_period))/np.arange(1, size + 1)

In [4]:
### SPAGHETTI PLOT OF THE MC SIMULATIONS OF THE EMPIRICAL GROWTH RATE OF THE RISKY ASSET

def generate_data_for_spaghetti_plot(params: Parameters, weights_vector: np.ndarray, num_simulations: int, size: int) -> tuple[pd.DataFrame, pd.DataFrame]:

    array_of_riskless_gross_returns = np.ones(size)*params.r

    results = []

    for _ in range(num_simulations):
        random_outcomes = np.random.choice([params.gamma_tails, params.gamma_heads], size=size, p=params.probabilities_vector)
        matrix_of_outcomes = np.column_stack((array_of_riskless_gross_returns, random_outcomes))
        results.append(return_running_emperical_growth_rates(np.dot(matrix_of_outcomes, weights_vector)))

    df_res = pd.DataFrame(np.array(results).T)
    df_augmented = df_res.copy()
    df_augmented['Fraction Positive'] = (df_res > 0).mean(axis=1)
    df_augmented['Fraction Negative'] = (df_res < 0).mean(axis=1)
    df_augmented['Net Diffusion Index'] = df_augmented['Fraction Positive'] - df_augmented['Fraction Negative']
    df_augmented['Mean'] = df_res.mean(axis=1)
    df_augmented['Median'] = df_res.median(axis=1)
    df_augmented['95% CI Lower'] = df_res.apply(lambda x: np.percentile(x, 2.5), axis=1)
    df_augmented['95% CI Upper'] = df_res.apply(lambda x: np.percentile(x, 97.5), axis=1)

    return df_augmented



def generate_spaghetti_plot(df_augmented: pd.DataFrame, weights_vector: np.ndarray, params: Parameters, title: str) -> go.Figure:

    golden_ratio = (1 + np.sqrt(5))/2

    asymptotic_avg = params.return_expected_log_portfolio_gross_return(weights_vector)
    path_cols = [c for c in df_augmented.columns if type(c) is not str]

    if asymptotic_avg > 0:
        position_lta = "top right"
        position_be = "bottom right"
        color_lta = "rgb(26, 150, 65)"
    else:
        position_lta = "bottom right"
        position_be = "top right"
        color_lta = "rgb(215, 48, 39)"


    fig = make_subplots(
        rows=2, 
        cols=1, 
        shared_xaxes=True,
        row_heights=[2, 1], 
        vertical_spacing=0.1
        )


    # Update figure layout
    fig.update_layout(
        template="plotly_white",
        title=f"{title} (N = {len(path_cols)})",
        yaxis_title="Empirical Growth Rate",
        xaxis2_title="Period",
        yaxis2_title="Fraction of Paths",
        height=500,
        width=500*golden_ratio,
    )    


    # Spaghetti plot of the MC simulations
    for c in path_cols:
        fig.add_trace(
            go.Scatter(
                x=df_augmented.index, 
                y=df_augmented[c], 
                mode='lines',
                line=dict(color="rgba(41,128,185,0.05)", width=0.6),
                hoverinfo='skip', 
                showlegend=False), 
            row=1, 
            col=1,
            )


    # Add 95% confidence interval as two separate traces, one for the upper bound and one for the lower bound
    fig.add_trace(
        go.Scatter(
            x=df_augmented.index, 
            y=df_augmented['95% CI Upper'], 
            mode='lines',
            line=dict(color="blue", width=0.6),
            hovertemplate=(
                "<b>Upper CI:</b> %{y:.3f}"       
                "<extra></extra>"             
                ),
            showlegend=False,
            name="95% CI Upper",
            ),
        row=1, 
        col=1,
        )
    fig.add_trace(
        go.Scatter(
            x=df_augmented.index,
            y=df_augmented['95% CI Lower'],
            mode='lines',
            line=dict(color="blue", width=0.6),
            hovertemplate=(
                "<b>Lower CI:</b> %{y:.3f}"
                "<extra></extra>"
                ),            
            showlegend=False,
            name="95% CI Lower",
            ), 
        row=1, 
        col=1,
        )


    # Asymptotic growth rate
    fig.add_hline(
        y=asymptotic_avg, 
        line_width=1.5,
        line_dash="dash",
        line_color=color_lta,
        annotation_text=f"Asymptotic Avg: {asymptotic_avg:.3f}",
        annotation_position=position_lta,
        row=1,
        col=1,
    )


    # Break-even line at 0.0
    fig.add_hline(
        y=0.0,
        line_width=1.5,
        line_dash="dash",
        line_color="rgb(40, 40, 40)",
        annotation_text="Break-even: 0.0",
        annotation_position=position_be,
        row=1,
        col=1,
    )



    ### LOWER PLOT ###

    # Plot of the fraction that are positive and the fraction that are negative, as two separate traces
    fig.add_trace(
        go.Scatter(
            x=df_augmented.index,
            y=df_augmented['Fraction Positive'],
            mode='lines',
            line=dict(color="rgba(26, 150, 65, 0.50)", width=1.2),
            hovertemplate=(
                "<b>Positive:</b> %{y:.1%}"
                "<extra></extra>"
                ),
            showlegend=False,
            ),
        row=2,
        col=1,
        )
    fig.add_trace(
        go.Scatter(
            x=df_augmented.index,
            y=df_augmented['Fraction Negative'],
            mode='lines',
            line=dict(color="rgba(215, 48, 39, 0.50)", width=1.2),
            hovertemplate=(
                "<b>Negative:</b> %{y:.1%}"
                "<extra></extra>"   
                ),            
            showlegend=False,
            ), 
        row=2, 
        col=1,
        )


    fig.update_layout(
        hovermode="x unified",
        )

    fig.update_xaxes(
        showspikes=True,
        spikemode="across",
        spikesnap="cursor",
        spikedash="dot",
        spikethickness=1,
        spikecolor="rgba(80,80,80,0.6)",
    )

    fig.update_yaxes(
        range=[0, 1],
        tickformat='.0%', 
        row=2, 
        col=1,
        )


    return fig 


In [5]:
# GENERATE DATA FOR THE PLOT OF THE ARITHMETIC VS GEOMETRIC RETURN AS A FUNCTION OF ALLOCATION TO THE RISKY ASSET

def generate_arithmetic_vs_geometric_data(params: Parameters) -> pd.DataFrame:

    output = []
    for f in np.linspace(0, 1, 1001):
        weights_vector = np.array([1 - f, f])
        arithmetic_gross_return = params.return_arithmetic_portfolio_gross_return(weights_vector) # E[S] = weights_vector . E[X]
        geometric_gross_return = params.return_geometric_portfolio_gross_return(weights_vector) # exp(E[log(S)])
        growth_rate = params.return_expected_log_portfolio_gross_return(weights_vector) # E[log(S)]
        growth_rate_ceiling = np.log(params.return_arithmetic_portfolio_gross_return(weights_vector)) # log(E[S]). This follows from Jensen's inequality.

        output.append((f, arithmetic_gross_return, geometric_gross_return, growth_rate, growth_rate_ceiling))
    df_arith_geo = pd.DataFrame(output, columns=['f', 'Arithmetic Gross Return', 'Geometric Gross Return', 'Growth Rate', 'Growth Rate Ceiling'])
    return df_arith_geo

def generate_arithmetic_vs_geometric_plot(df: pd.DataFrame, opt_result_for_CRP: opt.OptimizeResult) -> go.Figure:

    coords_for_star = (opt_result_for_CRP.x[1], np.exp(-opt_result_for_CRP.fun))

    fig = px.line(
        df,
        x="f",
        y=["Arithmetic Gross Return", "Geometric Gross Return"],
        markers=False,
        labels={
            "value": "",
            "variable": "",
            "f": "Fraction Invested in Risky Asset"
        },
        title="Arithmetic vs Geometric Gross Returns"
    )

    # Break-even line at 1.0 gross return
    fig.add_hline(
        y=1.0, 
        line_width=2.0,
        line_dash="dash",
        line_color="black",
        annotation_text="Break-even: 1.0",
        annotation_position="top right" # Puts text cleanly above the line on the right side
    )


    name_map = {
        "Arithmetic Gross Return": "Arithmetic",
        "Geometric Gross Return": "Geometric",
    }

    fig.for_each_trace(
        lambda trace: trace.update(name=name_map[trace.name])
    )


    # Put a star at the optimal fraction and its corresponding geometric gross return
    fig.add_trace(
        go.Scatter(
            x=[coords_for_star[0]], 
            y=[coords_for_star[1]], 
            mode='markers',
            marker=dict(color="red", size=12, symbol="star"),
            hoverinfo='skip',
            showlegend=False,
            zorder=5, # Puts the star on top of the lines
            # name="Optimal Fraction",
        )
    )

    fig.add_annotation(
        x=coords_for_star[0], y=coords_for_star[1], text=f"f* = {coords_for_star[0]:.3f},  geo = {coords_for_star[1]:.4f}",
        showarrow=True, arrowhead=3, standoff=8, ax=80, ay=-30,
        font=dict(size=12), bgcolor='rgba(255,255,255,0.6)')


    # Put an X at the zero of the growth rate function
    # fig.add_trace(
    #     go.Scatter(
    #         x=[opt_result_for_zeros.x[1]], 
    #         y=[np.exp(-opt_result_for_zeros.fun)], 
    #         mode='markers',
    #         marker=dict(color="blue", size=12, symbol="x"),
    #         hoverinfo='skip',
    #         showlegend=False,
    #         # name="Zero of Growth Rate",
    #      )
    # )   

    # fig.add_annotation(
    #     x=opt_result_for_zeros.x[1], y=np.exp(-opt_result_for_zeros.fun), text=f"Zero of Growth Rate: f = {opt_result_for_zeros.x[1]:.3f}",
    #     showarrow=True, arrowhead=3, standoff=8, ax=-80, ay=-30,
    #     font=dict(size=12), bgcolor='rgba(255,255,255,0.6)')


    fig.update_layout(
        template="plotly_white",
        hovermode="x unified"
    )

    fig.update_layout(width=800, height=500)

    return fig

In [6]:
# Set the parameters for the model

params = Parameters(gamma_heads=2.0, p=0.5, alpha=1.25, r=0.97)


In [7]:
# Determine the optimal allocation to the risky asset

opt_result_for_CRP = opt.minimize_scalar(
    lambda f: -params.return_growth_rate(np.array([1-f, f])), # We introduce the negative sign because we want to maximize the growth rate, but the minimize function minimizes the objective function.
    bounds=(0.0, 1.0),
    method='bounded'
    )    

In [8]:
opt_result_for_CRP

 message: Solution found.
 success: True
  status: 0
     fun: -0.012677299157080157
       x: 0.3800034063798751
     nit: 7
    nfev: 7

In [ ]:
# We will simulate the growth rate of a portfolio that is fully invested in the risky asset,
# which means that the weights vector is [0, 1].

weights_for_risky_asset = np.array([0.0, 1.0])
weights_for_optimal_CRP = opt_result_for_CRP.x
num_simulations = 200
size = 5000 # number of periods to simulate



In [ ]:
df_risky_asset = generate_data_for_spaghetti_plot(params, weights_for_risky_asset, num_simulations, size)
df_optimal_CRP = generate_data_for_spaghetti_plot(params, weights_for_optimal_CRP, num_simulations, size)
fig_risky_asset = generate_spaghetti_plot(df_risky_asset, weights_for_risky_asset, params, title="Simulating the Risky Asset")
fig_optimal_CRP = generate_spaghetti_plot(df_optimal_CRP, weights_for_optimal_CRP, params, title="Simulating the Optimal CRP")

In [ ]:
fig_risky_asset.show()

In [ ]:
df_arith_geo = generate_arithmetic_vs_geometric_data(params)
fig_arith_geo = generate_arithmetic_vs_geometric_plot(df_arith_geo, opt_result_for_CRP)

In [ ]:
fig_arith_geo.show()